# PULSE-v5.3 检查点消融与 4K / 8K 检索

从训练到 2,000 步的两份原始权重评测。三个干预共用同一份 PULSE 权重。输出保留逐题命中、逐窗口损失和配对区间。专用运行时完成或超时后释放 L4。

In [ ]:
from pathlib import Path
import hashlib, os, subprocess, sys, tempfile, time, zipfile
import torch
from google.colab import drive, runtime

if not torch.cuda.is_available() or "L4" not in torch.cuda.get_device_name():
    raise RuntimeError("请选择 NVIDIA L4 再运行。")
drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/Colab Notebooks/PULSE_AI")
source = project / "pulse_ai_src.zip"
with source.open("rb") as f:
    sha = hashlib.file_digest(f,"sha256").hexdigest()
if sha != "0c4258fa39670f2c0df7ce48ddcae3039b33068358f4d88fddb41f7ea7e3c6c9":
    raise RuntimeError("Drive 源码已变化。请核对版本后重新运行。")
work = Path(tempfile.mkdtemp(prefix="pulse-review-",dir="/content"))
with zipfile.ZipFile(source) as z:
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if not target.is_relative_to(work):
            raise ValueError(member.filename)
    z.extractall(work)
env = dict(os.environ, PYTHONPATH=str(work/"src")+os.pathsep+str(work))
print("Drive 源码 SHA-256",sha,"GPU",torch.cuda.get_device_name(),"PyTorch",torch.__version__,flush=True)

In [ ]:
(work/'review_v53.py').write_text('"""Paired checkpoint interventions and retrieval tests for the v5.3 source bundle."""\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport math\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\n\nfrom pulse.pulse_v4 import PULSEv4Config, PULSEv4Model\nfrom pulse.scalable_transformer import ScalableTransformerConfig, ScalableTransformerLM\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(".tmp")\n    tmp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")\n    os.replace(tmp, path)\n\n\ndef file_hash(path):\n    with open(path, "rb") as handle:\n        return hashlib.file_digest(handle, "sha256").hexdigest()\n\n\ndef make_models(preset, device):\n    return {\n        "pulse": PULSEv4Model(getattr(PULSEv4Config, preset)()).to(device),\n        "transformer": ScalableTransformerLM(getattr(ScalableTransformerConfig, preset)()).to(device),\n    }\n\n\ndef load_pair(run, device):\n    cfg = json.loads((run / "experiment_results.json").read_text(encoding="utf-8"))\n    if (cfg["config"]["preset"], cfg["config"]["pulse_version"]) != ("colab_30m", "v4"):\n        raise ValueError("Expected the 30M AttnRes v5.3 run (class version v4).")\n    pair = make_models("colab_30m", device)\n    records = {}\n    for name, net in pair.items():\n        ck = run / "checkpoints" / (name + "_latest.pt")\n        data = torch.load(ck, weights_only=True, map_location="cpu")\n        if data["step"] != 2000:\n            raise ValueError(f"{ck.name} contains step {data[\'step\']}, expected 2000")\n        net.load_state_dict(data["model_state_dict"], strict=True)\n        if name == "pulse":\n            net.fast_weights.load_latent_state(data["latent_mean"], data["latent_var"],\n                                              data["latent_count"], data["prior_centroids"])\n            if net.hippocampus.has_stored_keys:\n                raise ValueError("Expected a pre-edit checkpoint with an empty RLS fact bank.")\n        net.eval()\n        records[name] = {"sha256": file_hash(ck), "step": int(data["step"]),\n                         "parameters": net.num_parameters()}\n        del data\n    return pair, records\n\n\ndef logits(net, x, mode="full"):\n    if isinstance(net, ScalableTransformerLM):\n        return net(x, use_cache=False)[0]\n    return net(x, run_core=mode != "no_core", use_fast_weights=mode == "full",\n               return_state=False)["logits"]\n\n\ndef nll_rows(logit, y, cap=0.0):\n    parts = []\n    for i in range(0, y.size(1), 128):\n        z = logit[:, i:i+128].float()\n        if cap > 0:\n            z = cap * torch.tanh(z / cap)\n        parts.append(F.cross_entropy(z.transpose(1, 2), y[:, i:i+128], reduction="none"))\n    return torch.cat(parts, 1)\n\n\ndef boot_ci(array, stat=np.mean, seed=8109):\n    a = np.asarray(array, dtype=np.float64)\n    if a.size == 0:\n        return None\n    rng = np.random.default_rng(seed)\n    draws = rng.integers(len(a), size=(1000, len(a)))\n    return np.quantile(stat(a[draws], axis=1), [.025, .975]).tolist()\n\n\ndef auc(pos, neg):\n    # p(positive > negative) with half credit for ties.\n    a, b = np.asarray(pos), np.asarray(neg)\n    sorted_b = np.sort(b)\n    return float(np.mean((np.searchsorted(sorted_b, a, "left") +\n                          np.searchsorted(sorted_b, a, "right")) / (2 * len(b))))\n\n\ndef ranged_windows(stream, width, count, seed):\n    n = len(stream) // width\n    if n < count:\n        raise ValueError(f"Only {n} disjoint windows for requested count {count}")\n    starts = np.random.default_rng(seed).choice(n, count, replace=False) * width\n    return [stream[int(a):int(a)+width].long() for a in starts], starts.tolist()\n\n\ndef arms(pair):\n    return [(name, pair["pulse"], name) for name in ("full", "no_fast", "no_core")] + [\n        ("transformer", pair["transformer"], "full")]\n\n\n@torch.no_grad()\ndef perplexity(pair, stream, lengths, n, device):\n    windows, starts = ranged_windows(stream, max(lengths) + 1, n, 42003)\n    result = {"window_starts": starts, "by_length": {}}\n    for length in lengths:\n        rows = {}\n        for name, net, mode in arms(pair):\n            metrics = {"all": [], "common_suffix_128": [], "tail_after_512": []}\n            for window in windows:\n                row = window[-length-1:].to(device)[None, :]\n                z = logits(net, row[:, :-1], mode)\n                cap = net.config.logit_softcap if isinstance(net, PULSEv4Model) and length > 2048 else 0\n                loss = nll_rows(z, row[:, 1:], cap)\n                metrics["all"].append(loss.double().mean().item())\n                metrics["common_suffix_128"].append(loss[:, -min(128, length):].double().mean().item())\n                if length > 512:\n                    metrics["tail_after_512"].append(loss[:, 512:].double().mean().item())\n                del z, loss\n            rows[name] = {k: {"window_nll": v, "ppl": math.exp(float(np.mean(v)))}\n                          for k, v in metrics.items() if v}\n        result["by_length"][length] = {"arms": rows, "paired_nll_vs_full": {}}\n        for name in ("no_fast", "no_core", "transformer"):\n            for metric in rows["full"]:\n                d = np.subtract(rows[name][metric]["window_nll"], rows["full"][metric]["window_nll"])\n                result["by_length"][length]["paired_nll_vs_full"][name+"_"+metric] = {\n                    "delta_nll": float(d.mean()), "ci95": boot_ci(d),\n                    "ppl_ratio": math.exp(float(d.mean()))}\n    return result\n\n\n@torch.no_grad()\ndef common_scores(net, x, y, mode="no_core"):\n    t = y.size(1)\n    z = logits(net, torch.cat([x, y[:, :-1]], 1), mode)[:, -t:].float()\n    lp = z.log_softmax(-1)\n    nll = -lp.gather(-1, y[..., None]).squeeze(-1)\n    prob = lp.exp()\n    entropy = -(prob * lp).sum(-1)\n    sd = (prob * (-lp - entropy[..., None]).square()).sum(-1).clamp_min(.25).sqrt()\n    normalized = (nll - entropy) / sd\n    bos = torch.full_like(x[:, :1], min(net.config.vocab_size - 1, 50256))\n    u = logits(net, torch.cat([bos, y[:, :-1]], 1), "no_core").float().log_softmax(-1)\n    unll = -u.gather(-1, y[..., None]).squeeze(-1)\n    neg_pmi = (nll-unll).mean(1)\n    ch_a = .60 * normalized[:, :min(3,t)].mean(1) + 1.20 * neg_pmi\n    if t >= 7:\n        fc = torch.cat([nll[:, :3], nll[:, 6:]], 1).mean(1)\n        fu = torch.cat([unll[:, 1:3], unll[:, 6:]], 1).mean(1)\n        hat = ((nll[:, 3:6].mean(1)-fc) + .50*(unll[:, 3:6].mean(1)-fu)).relu()\n    elif t >= 3:\n        a, b = t//3, max(t//3+1, (2*t)//3)\n        hat = (nll[:, a:b].mean(1) - torch.cat([nll[:, :a], nll[:, b:]], 1).mean(1)).relu()\n    else:\n        hat = normalized.relu().amax(1)\n    ch_b = .50 * (nll.mean(1) + .35 * hat) - 5\n    shared = F.softplus(torch.logsumexp(torch.stack((2*ch_a, 2*ch_b)), 0)/2 + 2.5, beta=1.2)\n    return {"nll": nll.mean(1), "negative_pmi": neg_pmi, "legacy_ab": shared}\n\n\ndef honesty(pair, stream, n, device, encoder=None):\n    windows, _ = ranged_windows(stream, 136, n, 9021)\n    rows = torch.stack(windows).to(device)\n    src, truth = rows[:, :128], rows[:, 128:]\n    false = truth.roll(1, 0)\n    if n < 2:\n        raise ValueError("Cross-document pairing needs at least two windows.")\n    # Random-position token corruption is a corruption detector, not an entity benchmark.\n    g = torch.Generator().manual_seed(612)\n    pos = torch.randint(truth.size(1), (n,), generator=g).to(device)\n    i = torch.arange(n, device=device)\n    bad = truth.clone()\n    bad[i, pos] = false[i, pos]\n    valid_swap = bad.ne(truth).any(1)\n    sets = {"shuffled_continuation": (src, truth, false), "random_token": (\n        src[valid_swap], truth[valid_swap], bad[valid_swap])}\n    if encoder is not None:\n        names = ["Alice","David","Sarah","James","Peter","Julia","Laura","Henry"]\n        patterns = [("Today {a} carried the key, while {b} carried the book.", " The key was carried by {a}."),\n                    ("The letter was signed by {a}. The envelope was delivered by {b}.", " It was {a} who signed the letter."),\n                    ("At the meeting {b} thanked {a} for buying the tickets.", " The person who bought the tickets was {a}."),\n                    ("Of the two students, {b} took the bus and {a} rode the train.", " {a} was the one who rode the train.")]\n        triples = []\n        for j in range(min(n,224)):\n            a, b = names[j%8], names[(j%8+1+(j//32)%7)%8]\n            p, c = patterns[(j//8)%4]\n            texts = [p.format(a=a,b=b), c.format(a=a), c.format(a=b)]\n            triples.append(tuple(torch.tensor(encoder.encode_ordinary(s), device=device)[None] for s in texts))\n        sets["controlled_name_swap"] = triples\n    all_results = {}\n    for kind, items in sets.items():\n        triples = items if isinstance(items, list) else [\n            tuple(x[j:j+1] for x in items) for j in range(len(items[0])) if not torch.equal(items[1][j], items[2][j])]\n        if not triples:\n            continue\n        scores = {}\n        for name, net, mode in arms(pair):\n            acc = {}\n            for p, good, wrong in triples:\n                for label, cand in (("true",good),("false",wrong)):\n                    with torch.no_grad():\n                        out = common_scores(net, p, cand, mode)\n                        if name in ("full","no_fast"):\n                            out["native"] = net.continuation_free_energy(p, cand, use_fast_weights=name=="full")\n                    for metric, v in out.items():\n                        acc.setdefault(metric, {"true": [], "false": []})[label].append(float(v.item()))\n            scores[name] = {m: {**a,"auroc":auc(a["false"],a["true"])} for m,a in acc.items()}\n        refs = [("full","native","no_core","legacy_ab"),("full","native","no_fast","native"),\n                ("full","negative_pmi","transformer","negative_pmi"),\n                ("full","legacy_ab","transformer","legacy_ab")]\n        deltas = {}\n        rng = np.random.default_rng(1488)\n        indices = rng.integers(len(triples),size=(500,len(triples)))\n        for a,k,b,l in refs:\n            av,bv=scores[a][k],scores[b][l]\n            ap,an,bp,bn=[np.array(v) for v in (av["false"],av["true"],bv["false"],bv["true"])]\n            d=[auc(ap[idx],an[idx])-auc(bp[idx],bn[idx]) for idx in indices]\n            deltas[f"{a}.{k} minus {b}.{l}"]={"difference":av["auroc"]-bv["auroc"],\n                                               "paired_ci95":np.quantile(d,[.025,.975]).tolist()}\n        all_results[kind] = {"pairs":len(triples),"scores":scores,"comparisons":deltas}\n    return all_results\n\n\ndef mqar_sample(length, pairs, vocab, seed):\n    """Controlled multi-query AR variant. Query labels never occur in the query suffix."""\n    rng = np.random.default_rng(seed)\n    pool = min(512, (vocab-8)//3)\n    if not (0 < pairs <= pool and 4*pairs+4 < length):\n        raise ValueError("Invalid MQAR shape")\n    keys = rng.choice(pool, pairs, replace=False)+4\n    vals = rng.choice(pool, pairs, replace=False)+4+pool\n    x = rng.integers(4+2*pool, min(vocab, 4+3*pool), length)\n    # Disjoint pair slots and one last-quarter query per key, in a fresh order.\n    slots = np.sort(rng.choice(np.arange(0, 3*length//4-1, 2), pairs, replace=False))\n    qpos = np.sort(rng.choice(np.arange(3*length//4, length), pairs, replace=False))\n    order = rng.permutation(pairs)\n    x[slots], x[slots+1], x[qpos] = keys, vals, keys[order]\n    return torch.from_numpy(x), torch.from_numpy(qpos), torch.from_numpy(vals[order]), slots[order]\n\n\n@torch.no_grad()\ndef mqar(pair, lengths, loads, episodes, device):\n    report = {}\n    for length in lengths:\n        for load in loads:\n            if 4*load+4 >= length:\n                continue\n            cell = {k:[] for k in pair}\n            gaps = []\n            for j in range(episodes):\n                x, q, y, slots = mqar_sample(length, load, pair["pulse"].config.vocab_size, 410927+1009*j+load)\n                gaps.append((q.numpy()-slots).tolist())\n                for k,net in pair.items():\n                    z = logits(net, x[None].to(device), "no_core")\n                    guess = z[0,q.to(device)].argmax(-1).cpu()\n                    cell[k].append(guess.eq(y).tolist())\n                    del z\n            stats = {}\n            for k, rows in cell.items():\n                outcomes = np.asarray(rows)\n                stats[k] = {"hits":int(outcomes.sum()),"queries":int(outcomes.size),\n                            "accuracy":float(outcomes.mean()),"episode_ci95":boot_ci(outcomes.mean(1)),\n                            "all_queries_correct":float(outcomes.all(1).mean()),"hits_by_query":rows}\n            diff = np.array(cell["pulse"],int).mean(1)-np.array(cell["transformer"],int).mean(1)\n            report[f"{length}/{load}"] = {"models":stats,"gap_tokens":gaps,\n                                            "paired_delta":float(diff.mean()),"paired_ci95":boot_ci(diff)}\n    return report\n\n\ndef retrieval_fit(pair, steps, device):\n    # Common query-only supervision on short contexts. Never optimize on long test cases.\n    optim = {k:torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],lr=6e-4) for k,m in pair.items()}\n    for step in range(steps):\n        examples = [mqar_sample(256, 8, pair["pulse"].config.vocab_size, 12000000+16*step+i) for i in range(8)]\n        x,q,y = [torch.stack([e[j] for e in examples]).to(device) for j in range(3)]\n        for k,net in pair.items():\n            net.train()\n            optim[k].zero_grad(set_to_none=True)\n            z = logits(net, x, "no_core")\n            loss = F.cross_entropy(z[torch.arange(len(x),device=device)[:,None],q].flatten(0,1), y.flatten())\n            loss.backward()\n            nn.utils.clip_grad_norm_(net.parameters(),1)\n            optim[k].step()\n    for net in pair.values():\n        net.eval()\n\n\ndef passkeys(pair, stream, encoder, lengths, n, device):\n    import re\n    rng = np.random.default_rng(49334)\n    enc = encoder.encode_ordinary\n    question = enc("\\nWhat is the five-digit passkey? The passkey is")\n    result = {}\n    for length in lengths:\n        for depth in (.1,.5,.9):\n            outcomes = {k:[] for k in pair}\n            for j in range(n):\n                code = str(rng.integers(10000,100000))\n                msg = enc("\\nThe five-digit passkey is "+code+". Remember this passkey.\\n")\n                available = length-len(msg)-len(question)\n                if available <= 0:\n                    raise ValueError("Passkey context too short.")\n                start = int(rng.integers(0,len(stream)-available))\n                filler = stream[start:start+available].long().tolist()\n                at = int(depth*available)\n                prompt = torch.tensor(filler[:at]+msg+filler[at:]+question,device=device)[None]\n                for k,net in pair.items():\n                    tokens = []\n                    x = prompt\n                    for _ in range(8):\n                        with torch.no_grad():\n                            tok = int(logits(net,x,"no_core")[0,-1].argmax())\n                        tokens.append(tok)\n                        decoded = encoder.decode(tokens)\n                        if re.search(r"^\\s*\\d{5}\\D|^\\s*\\d{6}|^\\s*[^\\d\\s]",decoded):\n                            break\n                        x = torch.cat([x,torch.tensor([[tok]],device=device)],1)\n                    hit = bool(re.match(r"^\\s*"+code+r"(?!\\d)",encoder.decode(tokens)))\n                    outcomes[k].append(hit)\n            result[f"{length}/{depth}"]={k:{"correct":sum(v),"n":n,"accuracy":float(np.mean(v)),\n                "hits":v,"wilson95":wilson(sum(v),n)} for k,v in outcomes.items()}\n    return result\n\n\ndef wilson(hit,n):\n    z = 1.95996398454\n    p = hit / n\n    center = (p+z*z/(2*n))/(1+z*z/n)\n    margin = z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/(1+z*z/n)\n    return [max(0.,center-margin),min(1.,center+margin)]\n\n\nclass LowRank(nn.Module):\n    def __init__(self, base, rank):\n        super().__init__()\n        self.base = base\n        self.a = nn.Linear(base.in_features, rank, bias=False, device=base.weight.device)\n        self.b = nn.Linear(rank, base.out_features, bias=False, device=base.weight.device)\n        nn.init.zeros_(self.b.weight)\n\n    def forward(self,x):\n        return self.base(x)+self.b(self.a(x))\n\n\ndef install_lora(net, seed):\n    torch.manual_seed(seed)\n    net.requires_grad_(False)\n    # Every mixer has a width-by-width output projection. Identical ranks and sites at 30M/100M.\n    for block in net.blocks:\n        mixer = block if isinstance(net,PULSEv4Model) else block.attn\n        mixer.o_proj = LowRank(mixer.o_proj,8)\n    net.emb_out = LowRank(net.emb_out,8)\n    params=[p for p in net.parameters() if p.requires_grad]\n    return params\n\n\ndef lora_replay(net, opt, params, cue, target, replay, steps, old=None):\n    net.train()\n    for step in range(steps):\n        opt.zero_grad(set_to_none=True)\n        # One target per fact, just like the RLS bypass. Both LoRA arms see exactly the same replay.\n        facts=logits(net,cue,"no_core")[:,-1]\n        r=replay[step%len(replay)]\n        general=logits(net,r[:,:-1],"no_core")\n        edit_loss=F.cross_entropy(facts,target[:,0])\n        if old is not None:\n            ox,oy=old\n            old_loss=F.cross_entropy(logits(net,ox,"no_core")[:,-1],oy[:,0])\n            edit_loss=(edit_loss+old_loss)*.5\n        loss=.5*edit_loss+.5*F.cross_entropy(general.transpose(1,2),r[:,1:])\n        loss.backward()\n        nn.utils.clip_grad_norm_(params,1)\n        opt.step()\n    net.eval()\n\n\ndef edits(pair, stream, device, seed, n=12, steps=48):\n    k=pair["pulse"].config.semantic_clock\n    rng=np.random.default_rng(seed)\n    vocab=pair["pulse"].config.vocab_size\n    # Two waves. Base set is held out from edits AND from optimizer replay.\n    spans, _=ranged_windows(stream, 65, 2*n+24, seed)\n    cues=torch.stack([s[:2*k] for s in spans[:2*n]]).to(device)\n    targets=torch.tensor(rng.choice(np.arange(4,vocab),2*n,replace=False),device=device)[:,None]\n    replay=[torch.stack(spans[2*n+i:2*n+i+2]).to(device) for i in range(0,16,2)]\n    base=torch.stack(spans[2*n+16:]).to(device)\n    def measure(net,mode):\n        with torch.no_grad():\n            pred=logits(net,cues,mode)[:,-1].float()\n            base_nll=nll_rows(logits(net,base[:,:-1],mode),base[:,1:]).mean().item()\n        hit=pred.argmax(1).eq(targets[:,0]).cpu().tolist()\n        return {"old_top1":sum(hit[:n])/n,"new_top1":sum(hit[n:])/n,"hits":hit,\n                "target_nll":float(F.cross_entropy(pred,targets[:,0])),"base_ppl":math.exp(base_nll)}\n    result={}\n    native=copy.deepcopy(pair["pulse"])\n    stages={"pre":measure(native,"full")}\n    for wave in range(2):\n        for i in range(wave*n,(wave+1)*n):\n            native.teach_fact_online(cues[i:i+1],targets[i:i+1])\n        stages[f"after_wave_{wave+1}"]=measure(native,"full")\n    stages["remove_fast_after_teaching"]=measure(native,"no_fast")\n    stages["remove_core_after_teaching"]=measure(native,"no_core")\n    native.sleep_consolidate(num_replay_samples=48,clear_fast_weights=True)\n    stages["sleep_public_lm"]=measure(native,"full")\n    stages["sleep_gate_disabled"]=measure(native,"no_fast")\n    stages["fast_norm_after_sleep"]=float(native.hippocampus.w_fast.norm())\n    result["pulse_rls"]=stages\n    del native\n    for name, original in pair.items():\n        net=copy.deepcopy(original)\n        st={"pre":measure(net,"no_core")}\n        params=install_lora(net,seed)\n        opt=torch.optim.AdamW(params,lr=0.002,weight_decay=0)\n        st["adapter_parameters"]=sum(p.numel() for p in params)\n        for wave in range(2):\n            lo=wave*n\n            old=(cues[:n],targets[:n]) if wave else None\n            lora_replay(net,opt,params,cues[lo:lo+n],targets[lo:lo+n],replay,steps,old)\n            st[f"after_wave_{wave+1}"]=measure(net,"no_core")\n        result[name+"_lora_replay"]=st\n    return result\n\n\ndef backing_bytes(state):\n    stores={}\n    def walk(x):\n        if torch.is_tensor(x):\n            s=x.untyped_storage()\n            stores[(str(x.device),s.data_ptr())]=s.nbytes()\n        elif isinstance(x,dict):\n            for v in x.values(): walk(v)\n        elif isinstance(x,(list,tuple)):\n            for v in x: walk(v)\n    walk(state)\n    return sum(stores.values())\n\n\n@torch.no_grad()\ndef invariants(pair, lengths, device):\n    p=pair["pulse"]\n    result={}\n    for n in lengths:\n        g=torch.Generator().manual_seed(3191+n)\n        x=torch.randint(4,p.config.vocab_size,(1,n),generator=g).to(device)\n        out=p(x,use_fast_weights=True,run_core=True,return_state=True)\n        ref=out["logits"][:,[n//2-1,-1]].detach().clone()\n        state=out["working_state"]\n        record={"logical_bytes":p.working_state_bytes(state),"backing_bytes":backing_bytes(state),\n                "empty_bank":not p.hippocampus.has_stored_keys}\n        del out,state\n        for mode in ("no_fast","no_core"):\n            z=logits(p,x,mode)[:,[n//2-1,-1]]\n            record["max_logit_delta_"+mode]=float((ref-z).abs().max())\n            del z\n        for name,net in pair.items():\n            # Same shape and kernel path; change only the unseen suffix.\n            suffix=x.clone()\n            suffix[:,n//2:]=torch.randint(4,p.config.vocab_size,(1,n-n//2),generator=g).to(device)\n            original=logits(net,x,"no_core")[:,n//2-1].clone()\n            other=logits(net,suffix,"no_core")[:,n//2-1]\n            record[name+"_future_suffix_delta"]=float((original-other).abs().max())\n            del original,other\n        result[n]=record\n    return result\n\n\ndef run_audit(pair, stream, out, device, n=24, smoke=False, encoder=None, fit_steps=160):\n    out=Path(out)\n    lengths=(64,128) if smoke else (512,2048,4096,8192)\n    scenarios={"interventions":lambda:invariants(pair,lengths,device),\n               "ppl":lambda:perplexity(pair,stream,lengths,min(n,24),device),\n               "corruptions":lambda:honesty(pair,stream,n,device,encoder),\n               "mqar_zero_shot":lambda:mqar(pair,(64,128) if smoke else (256,512,4096,8192),\n                                            (4,8) if smoke else (8,32,64),n,device),\n               "edits":lambda:edits(pair,stream,device,40713,n=3 if smoke else 12,steps=2 if smoke else 48)}\n    if encoder is not None:\n        scenarios["passkey"]=lambda:passkeys(pair,stream,encoder,(512,4096,8192),n,device)\n    for name,task in scenarios.items():\n        print("Evaluating",name,flush=True)\n        save_json(out/(name+".json"),task())\n    if fit_steps:\n        retrieval_fit(pair,fit_steps,device)\n        save_json(out/"mqar_task_trained.json",{"short_train_steps":fit_steps,"train_context":256,\n            "load":8,"evaluation":mqar(pair,(256,512,4096,8192),(8,32,64),n,device)})\n\n\nif __name__=="__main__":\n    cli=argparse.ArgumentParser()\n    cli.add_argument("--run",type=Path,required=True)\n    cli.add_argument("--out",type=Path,required=True)\n    cli.add_argument("--episodes",type=int,default=24)\n    args=cli.parse_args()\n    import tiktoken\n    dev=torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    models,provenance=load_pair(args.run,dev)\n    val=args.run/"data_cache"/"val_stream_tinystories.pt"\n    save_json(args.out/"provenance.json",{"checkpoints":provenance,"validation_sha256":file_hash(val),\n                                         "device":str(dev),"torch":torch.__version__})\n    run_audit(models,torch.load(val,weights_only=True,map_location="cpu"),args.out,dev,\n              n=args.episodes,encoder=tiktoken.get_encoding("gpt2"))\n',encoding='utf-8')

In [ ]:
import json, importlib.util
out = project/"runs"/"review_v53_checkpoints_v1"
out.mkdir(exist_ok=True,parents=True)
receipt={"stage":"starting","gpu":torch.cuda.get_device_name(),"source_sha256":sha,"started_unix":time.time()}
def record():
    p=out/"status.json"
    tmp=p.with_suffix(".tmp")
    tmp.write_text(json.dumps(receipt,indent=2))
    os.replace(tmp,p)
record()
try:
    subprocess.run([sys.executable,"-m","pip","-q","install","tiktoken","einops"],check=True,timeout=240)
    if importlib.util.find_spec("fla") is None:
        subprocess.run([sys.executable,"-m","pip","-q","install","--no-deps","fla-core"],timeout=240)
    old = project/"runs"/"exp_pulse_v53_micro2k"
    receipt["stage"]="running"
    record()
    subprocess.run([sys.executable,"-u",str(work/"review_v53.py"),"--run",str(old),"--out",str(out)],
                   env=env,check=True,timeout=4500)
    receipt["stage"]="complete"
except BaseException as exc:
    receipt.update(stage="stopped",error=str(exc))
    raise
finally:
    receipt["ended_unix"]=time.time()
    record()
    print(out,receipt,flush=True)
    runtime.unassign()